# Apresentação dos resultados - CUDA

**Discente: Igor Sousa dos Santos Santana**
<br>
**Docente: Esbel Tomás Valero Orellana**
<br>
**Disciplina: Processamento Paralelo**
<br>

## A proposta

Este trabalho propõe o desenvolvimento e a análise de desempenho de um algoritmo para o cálculo de *Bounding Box* aplicado a conjuntos de dados multidimensionais.
<br>
A implementação é realizada integralmente na linguagem C (CUDA), focando em eficiência, paralelismo baseado em processamento paralelo em placas gráficas (NVIDIA).

### Objetivos Principais:

- **Estrutura de Dados Customizada:** Para gerenciar os dados de entrada, foi implementada uma estrutura chamada Tensor, que encapsula o ponteiro de dados e as informações de dimensionalidade, permitindo uma manipulação robusta de volumes de dados médicos ou científicos.
- **Implementação Sequencial:** Desenvolvimento de uma versão de referência executada em um único processo, garantindo a corretude do cálculo dos limites espaciais da região de interesse.
- **Implementação Paralela com MPI:** Integração da biblioteca MPI, Message Passing Interface, para particionar os dados e distribuir a carga de processamento entre múltiplos processos independentes, visando reduzir o tempo de execução em grandes volumes de dados.
    - Os testes computacionais foram realizados utilizando uma configuração estável de 16 processos paralelos;
    - O ambiente de execução consistiu em uma máquina dedicada de 16 núcleos físicos, permitindo o mapeamento nativo de um processo por núcleo da CPU;
    - Para a análise dos resultados deste trabalho, foram consolidados os dados de tempo e desempenho obtidos estritamente sob essa arquitetura de 16 processos.

### Estudo de Desempenho e Estratégias de Otimização com CUDA:

O foco central é a comparação entre diferentes estratégias de mapeamento de threads, organização da malha de computação (grid) e gerenciamento de memória fornecidos pela arquitetura CUDA. O objetivo é identificar qual abordagem de particionamento do volume e consolidação de resultados melhor se adapta à carga de trabalho do algoritmo de Bounding Box sobre a estrutura de Tensor 3D. Estão sendo avaliadas as seguintes abordagens:
- **Sequencial (CPU Host):** Execução isolada e linear no processador principal, CPU, sem paralelismo, servindo como linha de base para as métricas de aceleração;
- Mapeamento Estático de Threads (Mapeamento Direto): Divisão uniforme e contígua das dimensões do Tensor em blocos tridimensionais de threads (8×8×8). Cada thread se encarrega de computar um único elemento da matriz na memória global, minimizando a complexidade do código de indexação;
- **Fusão de Kernels (Kernel Fusion):** Amalgamação de múltiplas etapas do pipeline, binarização e extração de índices, em um único disparo de kernel. Essa estratégia visa mitigar o custo de acessos repetitivos à memória global da VRAM e eliminar o overhead de múltiplos lançamentos de funções pela CPU;
- **Redução por Operações Atômicas e Controle de Fluxo:** Utilização de instruções nativas de hardware (como atomicMin e atomicMax) integradas a kernels dedicados de controle para consolidar as coordenadas locais da *Bounding Box* diretamente na GPU. Essa abordagem elimina a necessidade de transferências intermediárias redundantes de dados através do barramento PCIe.

Ao final desta análise, os resultados de tempo de execução, speedup, ocupação da GPU e eficiência na largura de banda da memória serão confrontados para determinar a melhor configuração de execução paralela (geometria de blocos e grades) e a topologia de otimização de memória ideal para a estrutura de Tensor proposta.

## Implementação

Nesse capítulo serão apresentados os códigos fonte em C (CUDA) e Python utilizados para gerar os dados explorados nesse trabalho.
<br>
O código C foi exportado para uma biblioteca dinamica (.so) para uso direto em conjunção com os códigos da linguagem Python.

### C (CUDA)

```c
#include <ctime>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#ifdef _WIN32
#define EXPORT __declspec(dllexport)
#else
#define EXPORT
#endif

#ifdef __cplusplus
extern "C" {
#endif

EXPORT __device__ void get_indices_with_padding(int coordinates[6], const int i, const int j, const int k) {
    atomicMin(&coordinates[0], i);
    atomicMin(&coordinates[1], j);
    atomicMin(&coordinates[2], k);

    atomicMax(&coordinates[3], i);
    atomicMax(&coordinates[4], j);
    atomicMax(&coordinates[5], k);
    return;
}

EXPORT __global__ void binarize_data(
    double *data_ptr,
    const int x_axis,
    const int y_axis,
    const int z_axis,
    const double THRESHOLD,
    int coordinates[6]
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x,
        j = blockIdx.y * blockDim.y + threadIdx.y,
        k = blockIdx.z * blockDim.z + threadIdx.z;

    if(i < x_axis && j < y_axis && k < z_axis) {
        int index = (i * y_axis * z_axis) + (j * z_axis) + k;

        if(data_ptr[index] > THRESHOLD) {
            data_ptr[index] = 1.0;
            get_indices_with_padding(coordinates, i, j, k);
        } else {
            data_ptr[index] = 0.0;
        }
    }

    return;
}

EXPORT __global__ void apply_padding_and_dimensions(
    int coordinates[6],
    const int x_axis,
    const int y_axis,
    const int z_axis,
    int *output_dimensions
) {
    if(blockIdx.x == 0 && threadIdx.x == 0) {
        if(coordinates[3] < coordinates[0]) {
            output_dimensions[0] = 0;
            return;
        }

        coordinates[0] = (coordinates[0] > 5) ? coordinates[0] - 5 : 0;
        coordinates[1] = (coordinates[1] > 5) ? coordinates[1] - 5 : 0;
        coordinates[2] = (coordinates[2] > 5) ? coordinates[2] - 5 : 0;
        coordinates[3] = (coordinates[3] + 5 < x_axis) ? coordinates[3] + 5 : x_axis - 1;
        coordinates[4] = (coordinates[4] + 5 < y_axis) ? coordinates[4] + 5 : y_axis - 1;
        coordinates[5] = (coordinates[5] + 5 < z_axis) ? coordinates[5] + 5 : z_axis - 1;

        output_dimensions[0] = (coordinates[3] - coordinates[0] + 1);
        output_dimensions[1] = (coordinates[4] - coordinates[1] + 1);
        output_dimensions[2] = (coordinates[5] - coordinates[2] + 1);
    }

    return;
}

EXPORT __global__ void crop_data(
    double *data_ptr,
    const int x_axis,
    const int y_axis,
    const int z_axis,
    const int coordinates[6],
    double *output_data_ptr,
    int x_output_axis,
    int y_output_axis,
    int z_output_axis
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x,
        j = blockIdx.y * blockDim.y + threadIdx.y,
        k = blockIdx.z * blockDim.z + threadIdx.z;

    if(i < x_output_axis && j < y_output_axis && k < z_output_axis) {
        int input_i = i + coordinates[0],
            input_j = j + coordinates[1],
            input_k = k + coordinates[2],
            input_index = (input_i * y_axis * z_axis) + (input_j * z_axis) + input_k,
            output_index = (i * y_output_axis * z_output_axis) + (j * z_output_axis) + k;

        output_data_ptr[output_index] = data_ptr[input_index];
    }

    return;
}

EXPORT __host__ void bounding_box_pipeline(
    double *host_input_data_ptr,
    const int x_axis,
    const int y_axis,
    const int z_axis,
    const double THRESHOLD,
    double *host_output_data_ptr
) {
    double *device_input_data = NULL,
        *device_output_data = NULL;
    int host_coordinates[6] = {x_axis, y_axis, z_axis, -1, -1, -1},
        *device_coordinates = NULL,
        host_output_dimensions[3] = {0, 0, 0},
        *device_output_dimensions = NULL,
        x_output_axis,
        y_output_axis,
        z_output_axis;
    dim3 threads_in_block(8, 8, 8),
        number_of_blocks((x_axis + threads_in_block.x - 1) / threads_in_block.x,
                         (y_axis + threads_in_block.y - 1) / threads_in_block.y,
                         (z_axis + threads_in_block.z - 1) / threads_in_block.z);
    time_t start,
        total_time = clock();

    cudaMalloc((void**) &device_input_data, x_axis * y_axis * z_axis * sizeof(double));
    cudaMalloc((void**) &device_coordinates, 6 * sizeof(int));
    cudaMalloc((void**) &device_output_dimensions, 3 * sizeof(int));

    cudaMemcpy(device_input_data, host_input_data_ptr, x_axis * y_axis * z_axis * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(device_coordinates, host_coordinates, 6 * sizeof(int), cudaMemcpyHostToDevice);

    start = clock();
    binarize_data<<<number_of_blocks, threads_in_block>>>(device_input_data, x_axis, y_axis, z_axis, THRESHOLD, device_coordinates);
    cudaDeviceSynchronize();
    printf("Tempo (binarize_data): %.3lf\n", (double) (clock() - start) / CLOCKS_PER_SEC);
    start = clock();
    apply_padding_and_dimensions<<<1, 1>>>(device_coordinates, x_axis, y_axis, z_axis, device_output_dimensions);
    printf("Tempo (apply_padding_and_dimensions): %.3lf\n", (double) (clock() - start) / CLOCKS_PER_SEC);

    cudaMemcpy(host_output_dimensions, device_output_dimensions, 3 * sizeof(int), cudaMemcpyDeviceToHost);

    if(!host_output_dimensions[0]) {
        printf("Error!\nNo data was found.\n");
        cudaFree(device_input_data);
        cudaFree(device_coordinates);
        cudaFree(device_output_dimensions);
        return;
    }

    x_output_axis = host_output_dimensions[0];
    y_output_axis = host_output_dimensions[1];
    z_output_axis = host_output_dimensions[2];
    cudaMalloc((void**) &device_output_data, x_output_axis * y_output_axis * z_output_axis * sizeof(double));

    dim3 number_of_blocks_crop((x_output_axis + threads_in_block.x - 1) / threads_in_block.x,
                               (y_output_axis + threads_in_block.y - 1) / threads_in_block.y,
                               (z_output_axis + threads_in_block.z - 1) / threads_in_block.z);

    start = clock();
    crop_data<<<number_of_blocks_crop, threads_in_block>>>(
        device_input_data,
        x_axis,
        y_axis,
        z_axis,
        device_coordinates,
        device_output_data,
        x_output_axis,
        y_output_axis,
        z_output_axis
    );
    cudaDeviceSynchronize();
    printf("Tempo (crop_data): %.3lf\n", (double) (clock() - start) / CLOCKS_PER_SEC);

    cudaMemcpy(
        host_output_data_ptr,
        device_output_data,
        x_output_axis * y_output_axis * z_output_axis * sizeof(double),
        cudaMemcpyDeviceToHost
    );
    printf("Tempo total (C): %.3lf\n", (double) (clock() - start) / CLOCKS_PER_SEC);

    // desalocando tudo
    cudaFree(device_input_data);
    cudaFree(device_output_data);
    cudaFree(device_coordinates);
    cudaFree(device_output_dimensions);
    return;
}

#ifdef __cplusplus
}
#endif
```

### Python (Jupyter Notebook)

#### Gerando *script* de execução

In [ ]:
%%writefile GenerateOutputCUDA.py
from typing import Any
import nibabel as nib
import pandas as pd
import numpy as np
import ctypes
import time

library: ctypes.CDLL = ctypes.CDLL("../bin/cuda_c.so")

library.bounding_box_pipeline.argtypes = [
    ctypes.POINTER(ctypes.c_double),
    ctypes.c_int,
    ctypes.c_int,
    ctypes.c_int,
    ctypes.c_double,
    ctypes.POINTER(ctypes.c_double)
]
library.bounding_box_pipeline.restype = None

# criando as variaveis
images_list: list[int] = [120, 205, 510, 650, 980]
nii_image_path: str = f"../images/{images_list[0]}"
image: np.ndarray = nib.load(f"{nii_image_path}.label.nii.gz").get_fdata()
crop_data: np.ndarray = np.empty(image.shape[0] * image.shape[1] * image.shape[2],
                                 dtype=np.float64)
# ---
del images_list
del nii_image_path
# ---
input_double_ptr: ctypes.POINTER = image.ctypes.data_as(
    ctypes.POINTER(ctypes.c_double)
)
output_double_ptr: ctypes.POINTER = crop_data.ctypes.data_as(
    ctypes.POINTER(ctypes.c_double)
)
x: ctypes.c_int = ctypes.c_int(image.shape[0])
y: ctypes.c_int = ctypes.c_int(image.shape[1])
z: ctypes.c_int = ctypes.c_int(image.shape[2])
threshold: ctypes.c_double = ctypes.c_double(0.5)
# esse output_size foi retirado dos trabalhos anteriores (OpenMP e MPI)
output_size: tuple[int, int, int] = (314, 249, 182)

# chamando a funcao
tempo_inicial: float = time.perf_counter()

library.bounding_box_pipeline(input_double_ptr,
                              x,
                              y,
                              z,
                              threshold,
                              output_double_ptr)
print(f"Tempo de execução (Python): {np.round(time.perf_counter() - tempo_inicial, 3)}s")

# trazendo o resultado de volta para numpy
crop_data = np.ctypeslib.as_array(output_double_ptr,
                                  output_size)

#### Executando o *script* gerado acima

A saída do script será salva em `presentation/data/` com nome `log_{iteração}.ncu-rep` 

Antes de executar o bloco acima, foi necessário executar ativar o `Profiler (NVIDIA Nsight Compute)`.
<br>
Para ativá-lo, foi necessário executar os comando abaixo no terminal linux.
<br>
```sh
echo 'options nvidia "NVreg_RestrictProfilingToAdminUsers=0"' | sudo tee /etc/modprobe.d/ncu-profiling.conf
sudo update-initramfs -c -k all
sudo reboot
system76-power graphics nvidia
```
<br>
Esses comando foram executados no Pop!_OS 24.04 LTS.

- Kernel versão 7.0.11-76070011-generic

#### Removendo o arquivo temporário `GenerateOutputCUDA.py`.

In [ ]:
!rm GenerateOutputCUDA.py